# Final Model Selection & Justification

###  Objective:
To build effective, interpretable, and scalable screening models for:
- **High Blood Pressure**
- **Diabetes**
- **Cardiovascular Condition (Heart disease or stroke)**

These models are designed for **screening purposes**, not diagnosis — meaning we care most about **identifying at-risk individuals early**, even at the cost of some false positives.

---

### Evaluation Approach:
We evaluated a diverse set of classification models:
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- Neural Network (MLP)

Each model was tested with:
- **Undersampling**
- **SMOTE Oversampling**

Performance was measured using:
- `Recall (Yes)` — to prioritize **identifying as many actual positive cases as possible**
- `F1 Score (Yes)` — to ensure recall isn’t misleading due to extremely low precision
- `ROC AUC` — to assess the model’s ability to **distinguish between at-risk and not-at-risk individuals**

---

###  Metric Prioritization Logic

We followed a **hierarchical, medically informed** evaluation strategy:

1. **Recall (Yes)** — our top priority, as **missing a positive case** in a screening context can be dangerous.
2. **F1 Score (Yes)** — used to ensure that high recall isn’t coming at the cost of extreme false positives.
3. **ROC AUC** — used as a support metric to assess the model’s ability to separate positive from negative cases, regardless of threshold.

This strategy ensures that we **catch true cases**, maintain **balanced trade-offs**, and build models that can scale for **public health use**.

---

###  Analysis Results:
Across all targets, **Logistic Regression (with Undersampling)** consistently delivered:
- **Strong Recall** across all three target variables
- **Balanced F1 Score** indicating a good trade-off with precision
- **High ROC AUC**, showing reliable separation between positive and negative classes

It also offered:
- Simplicity, fast training, and inference
- Clear feature interpretability
- A single unified pipeline across all models

---

###  Final Model Choice

We selected **Logistic Regression with Undersampling** for all three conditions based on empirical performance and practical advantages:

| Condition     | Final Model Selected              | Justification |
|---------------|-----------------------------------|------------------------------|
| High BP       | Logistic Regression (Undersample) | Demonstrated strong recall and balanced F1, with high AUC — ideal for early screening. |
| Diabetes      | Logistic Regression (Undersample) | Delivered high recall and the strongest AUC, while maintaining reliable F1 performance. |
| Cardio        | Logistic Regression (Undersample) | Achieved high recall and the highest AUC, with consistent F1 — supports risk stratification. |

>  This approach ensures **clinical relevance**, **consistency**, and **deployment simplicity** without sacrificing performance.

---

###  Next Step:
We will now:
- Retrain final models using **Logistic Regression (Undersampling)** for each target
- Package into a **unified, reusable pipeline**
- Generate **feature importance plots (coefficients)** to support explainability
- Prepare models for **integration into screening tools or apps**

This unified and interpretable approach maximizes **trust, scalability, and real-world value** in healthcare screening systems.


In [2]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

part14_metrics_dir = os.path.join(METRICS_DIR, "part14_final_models")
os.makedirs(part14_metrics_dir, exist_ok=True)

part14_stats_dir = os.path.join(STATS_DIR, "part14_final_models")
os.makedirs(part14_stats_dir, exist_ok=True)



In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Training Function
def train_and_save_model(data_path, target_variable, model_filename, preprocessor_filename, test_csv_filename):
    print(f"\n Training and Saving model for: {target_variable}")

    # Load and filter
    df = pd.read_csv(data_path)
    df = df[df[target_variable] != "Unknown"]

    X = df.drop(columns=[target_variable])
    y = df[target_variable]

    # Split into train and test (test remains untouched/unbalanced)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    # Preprocessor
    categorical_features = X.columns.tolist()
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

    # Model pipeline (with undersampling only on training)
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("undersample", RandomUnderSampler(random_state=42)),
        ("model", LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=42))
    ])

    # Fit on training data
    pipe.fit(X_train, y_train)

    # Save model and preprocessor
    joblib.dump(pipe, model_filename)
    joblib.dump(preprocessor, preprocessor_filename)
    print(f" Saved model: {model_filename}")
    print(f" Saved preprocessor: {preprocessor_filename}")

    # Save test data (original distribution) for ROC/Threshold use
    test_df = X_test.copy()
    test_df[target_variable] = y_test
    test_df.to_csv(test_csv_filename, index=False)
    print(f" Saved held-out test set: {test_csv_filename}")

# Run for all three models


# High blood pressure
train_and_save_model(
    data_path=os.path.join(STATS_DIR, "model1_high_bp.csv"),
    target_variable="Has a high blood pressure",
    model_filename=os.path.join(METRICS_DIR, "part14_final_models", "model_highbp.pkl"),
    preprocessor_filename=os.path.join(METRICS_DIR, "part14_final_models", "preprocessor_highbp.pkl"),
    test_csv_filename=os.path.join(STATS_DIR, "part14_final_models", "test_highbp.csv")
)

# Diabetes
train_and_save_model(
    data_path=os.path.join(STATS_DIR, "model2_diabetes.csv"),
    target_variable="Has diabetes",
    model_filename=os.path.join(METRICS_DIR, "part14_final_models", "model_diabetes.pkl"),
    preprocessor_filename=os.path.join(METRICS_DIR, "part14_final_models", "preprocessor_diabetes.pkl"),
    test_csv_filename=os.path.join(STATS_DIR, "part14_final_models", "test_diabetes.csv")
)

# Cardiovascular
train_and_save_model(
    data_path=os.path.join(STATS_DIR, "model3_cardio.csv"),
    target_variable="Cardiovascular condition (Heart disease or stroke)",
    model_filename=os.path.join(METRICS_DIR, "part14_final_models", "model_cardio.pkl"),
    preprocessor_filename=os.path.join(METRICS_DIR, "part14_final_models", "preprocessor_cardio.pkl"),
    test_csv_filename=os.path.join(STATS_DIR, "part14_final_models", "test_cardio.csv")
)



 Training and Saving model for: Has a high blood pressure
 Saved model: d:\Projects\health-risk-prediction\outputs\metrics\part14_final_models\model_highbp.pkl
 Saved preprocessor: d:\Projects\health-risk-prediction\outputs\metrics\part14_final_models\preprocessor_highbp.pkl
 Saved held-out test set: d:\Projects\health-risk-prediction\outputs\statistics\part14_final_models\test_highbp.csv

 Training and Saving model for: Has diabetes
 Saved model: d:\Projects\health-risk-prediction\outputs\metrics\part14_final_models\model_diabetes.pkl
 Saved preprocessor: d:\Projects\health-risk-prediction\outputs\metrics\part14_final_models\preprocessor_diabetes.pkl
 Saved held-out test set: d:\Projects\health-risk-prediction\outputs\statistics\part14_final_models\test_diabetes.csv

 Training and Saving model for: Cardiovascular condition (Heart disease or stroke)
 Saved model: d:\Projects\health-risk-prediction\outputs\metrics\part14_final_models\model_cardio.pkl
 Saved preprocessor: d:\Projects\he